## Model Comparison — Wildfire Risk Tier Classification

This notebook trains and evaluates multiple classifiers on the same feature matrix and train/test split used in `model_trainning.ipynb`, then compares their performance head-to-head to justify the choice of Random Forest.

**Models compared:**
1. Logistic Regression (baseline)
2. K-Nearest Neighbors
3. Decision Tree
4. Support Vector Machine (RBF kernel)
5. **Random Forest** (chosen model)
6. XGBoost
7. LightGBM

**Primary metric:** High-risk recall (class 2) — missing a high-risk tract is a safety failure.

**Secondary metrics:** F1 (macro), F1 (weighted), Accuracy, ROC-AUC (OvR macro).

## 1. Imports

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score,
    roc_auc_score, classification_report, confusion_matrix,
)

DATA_PROC = Path("../data/processed")
TIER_LABELS = {0: "Low", 1: "Medium", 2: "High"}

## 2. Load Features — Same Split as model_trainning.ipynb

In [ ]:
tracts = gpd.read_file(DATA_PROC / "features.geojson")
tracts = tracts.dropna(subset=["risk_tier"]).reset_index(drop=True)

FEATURES = [
    "dist_fire_station_m",
    "dist_hospital_m",
    "hydrant_density",
    "road_density",
    "pop_density",
    "rpl_theme_1",
    "rpl_theme_2",
    "rpl_theme_3",
    "rpl_theme_4",
]

X = tracts[FEATURES].copy()
y = tracts["risk_tier"].astype(int)

# Same seed and stratification as model_trainning.ipynb
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Total tracts : {len(X)}")
print(f"Train        : {len(X_train)}")
print(f"Test         : {len(X_test)}")
print(f"\nClass distribution (test):")
print(y_test.value_counts().sort_index().rename(TIER_LABELS))

## 3. Define Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    ),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=9),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=10, class_weight="balanced", random_state=42
    ),
    "SVM (RBF)": SVC(
        kernel="rbf", class_weight="balanced", probability=True, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=15, min_samples_split=2,
        class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        use_label_encoder=False, eval_metric="mlogloss",
        random_state=42, n_jobs=-1, verbosity=0
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=200, max_depth=10, learning_rate=0.1,
        class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1
    ),
}

print(f"Models to compare: {list(models.keys())}")

## 4. Train & Evaluate All Models

In [ ]:
results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    y_prob = model.predict_proba(X_test_sc)

    acc      = accuracy_score(y_test, y_pred)
    f1_w     = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    f1_m     = f1_score(y_test, y_pred, average="macro",    zero_division=0)
    rec_all  = recall_score(y_test, y_pred, average=None,  zero_division=0)
    rec_high = rec_all[2]

    y_bin   = label_binarize(y_test, classes=[0, 1, 2])
    roc_auc = roc_auc_score(y_bin, y_prob, multi_class="ovr", average="macro")

    cv_f1 = cross_val_score(model, scaler.fit_transform(X), y, cv=cv, scoring="f1_weighted", n_jobs=-1)

    results.append({
        "Model":           name,
        "Accuracy":        round(acc,      3),
        "F1 (weighted)":   round(f1_w,     3),
        "F1 (macro)":      round(f1_m,     3),
        "ROC-AUC":         round(roc_auc,  3),
        "High-Risk Recall":round(rec_high, 3),
        "CV F1 (mean)":    round(cv_f1.mean(), 3),
        "CV F1 (std)":     round(cv_f1.std(),  3),
    })
    print(f"{name:22s}  acc={acc:.3f}  f1_w={f1_w:.3f}  f1_m={f1_m:.3f}  "
          f"roc={roc_auc:.3f}  high_rec={rec_high:.3f}  "
          f"cv={cv_f1.mean():.3f}±{cv_f1.std():.3f}")

df = pd.DataFrame(results).set_index("Model")
print("\n=== Results Summary ===")
print(df.to_string())

## 5. Results Table

In [ ]:
df_display = df.copy()

# Highlight best value per column in bold using Styler
def highlight_max(s):
    is_max = s == s.max()
    return ["font-weight: bold; background-color: #d4edda" if v else "" for v in is_max]

df_display.drop(columns=["CV F1 (std)"]).style \
    .apply(highlight_max, axis=0) \
    .set_caption("Model Comparison — Test-Set Metrics (bold = best per column)")

## 6. Metric Comparison Charts

In [ ]:
metrics = ["Accuracy", "F1 (weighted)", "F1 (macro)", "ROC-AUC", "High-Risk Recall", "CV F1 (mean)"]
colors  = ["#4472C4" if m != "Random Forest" else "#C00000" for m in df.index]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, metric in zip(axes, metrics):
    vals = df[metric]
    bar_colors = ["#C00000" if m == "Random Forest" else "#4472C4" for m in vals.index]
    bars = ax.barh(vals.index, vals.values, color=bar_colors)
    ax.set_xlim(vals.min() - 0.05, 1.0)
    ax.set_title(metric, fontweight="bold")
    ax.set_xlabel("Score")
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
                f"{val:.3f}", va="center", fontsize=8)
    ax.axvline(vals.max(), color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

rf_patch   = mpatches.Patch(color="#C00000", label="Random Forest (chosen model)")
other_patch = mpatches.Patch(color="#4472C4", label="Other models")
fig.legend(handles=[rf_patch, other_patch], loc="lower center", ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.02))

plt.suptitle("Wildfire Risk Tier — Model Comparison", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()